In [1]:
# ==========================================================
# BLOQUE 1. IMPORTS Y MONTAJE DE DRIVE
#
# Solo se ejecuta una vez.
# ==========================================================

import os
import re
import json
import time
import unicodedata
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==========================================================
# BLOQUE 2. RUTA DEL JSON EN EL DIRECTORIO EN EL QUE ESTAMOS
# ==========================================================

NOMBRE_PROGRAMA = "Extrae_Rankings.ipynb"  # ajusta al nombre real de tu notebook


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(

    ruta_programa,

    "JSONs"

)

os.makedirs(

    CARPETA_JSON,

    exist_ok=True

)

In [3]:
# ==========================================================
# BLOQUE 3. CONFIGURACIÓN DE LA FUENTE
#
# Fuente: La UPV en los rankings
#
# A diferencia de "Servicios universitarios" (un listado con
# buscador), esta página es estática y sin JS de carga
# adicional: es una sucesión de "tarjetas" de ranking, cada
# una con un título, un enlace a la noticia UPV que lo
# desarrolla, un párrafo descriptivo y un enlace externo a
# la fuente del ranking (que abre en nueva ventana).
# ==========================================================

URL_RANKINGS = "https://www.upv.es/rankings/index.html"

NOMBRE_JSON = "rankings.json"

RUTA_JSON = os.path.join(
    CARPETA_JSON,
    NOMBRE_JSON
)


# ----------------------------------------------------------
# Cabeceras HTTP (idénticas a las de institución/servicios)
# ----------------------------------------------------------

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/avif,image/webp,"
        "*/*;q=0.8"
    ),
    "Accept-Language": (
        "es-ES,es;q=0.9,en;q=0.8"
    ),
    "Connection": "keep-alive"
}


# ==========================================================
# FUNCIONES AUXILIARES (idénticas a las de institución/servicios)
# ==========================================================

def limpiar_texto(texto):

    if texto is None:
        return ""

    texto = str(texto)

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


def normalizar_identificador(texto):

    texto = limpiar_texto(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")


def normalizar_url(
    url,
    url_base=URL_RANKINGS
):

    if not url:
        return ""

    url = url.strip()

    return urljoin(
        url_base,
        url
    )


def es_url_valida(url):

    if not url:
        return False

    try:

        analisis = urlparse(url)

        return analisis.scheme in {
            "http",
            "https"
        }

    except Exception:

        return False


def deduplicar_lista(
    elementos,
    clave="url"
):

    resultado = []
    vistos = set()

    for elemento in elementos:

        if not isinstance(
            elemento,
            dict
        ):
            continue

        valor = elemento.get(
            clave,
            ""
        )

        if valor in vistos:
            continue

        vistos.add(valor)

        resultado.append(
            elemento
        )

    return resultado


def crear_seccion(
    titulo,
    tipo="seccion"
):

    return {

        "id": normalizar_identificador(
            titulo
        ),

        "titulo": limpiar_texto(
            titulo
        ),

        "tipo": tipo,

        "descripcion": "",

        "elementos": []
    }


def crear_elemento(
    titulo="",
    descripcion="",
    url="",
    tipo="recurso",
    url_externa=""
):
    """
    Igual que en Servicios, con un campo extra opcional
    'url_externa': el enlace a la web propia del ranking
    (Shanghai Ranking, QS, THE...), que NO se scrapea (es
    un dominio externo) pero interesa conservar como
    referencia/cita.
    """

    elemento = {

        "tipo": tipo,

        "titulo": limpiar_texto(
            titulo
        ),

        "descripcion": limpiar_texto(
            descripcion
        ),

        "url": normalizar_url(
            url
        )
    }

    if url_externa:

        elemento["url_externa"] = normalizar_url(
            url_externa,
            URL_RANKINGS
        )

    return elemento


# ==========================================================
# ESTRUCTURA BASE DEL JSON
# ==========================================================

def crear_json_base():

    return {

        "titulo": "La UPV en los rankings",

        "url": URL_RANKINGS,

        "tipo": "padre",

        "secciones": [

            crear_seccion(
                "Rankings",
                "rankings"
            )
        ]
    }


json_rankings = crear_json_base()


# ==========================================================
# RUTAS DE MARKDOWN
# ==========================================================

CARPETA_RANKINGS = os.path.join(
    ruta_programa,
    "RANKINGS"
)

CARPETA_RANKINGS_RECURSOS = os.path.join(
    CARPETA_RANKINGS,
    "rankings"
)

os.makedirs(
    CARPETA_RANKINGS_RECURSOS,
    exist_ok=True
)

RUTA_MARKDOWN_PADRE_RANKINGS = os.path.join(
    CARPETA_RANKINGS,
    "rankings.md"
)


print()
print("=" * 70)
print("ESTRUCTURA DE MARKDOWN - RANKINGS")
print("=" * 70)

print()
print("Carpeta principal:")
print(CARPETA_RANKINGS)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE_RANKINGS)

print()
print("Carpeta de recursos:")
print(CARPETA_RANKINGS_RECURSOS)

print()
print("OK: estructura de directorios preparada.")

print()
print("=" * 70)


ESTRUCTURA DE MARKDOWN - RANKINGS

Carpeta principal:
/content/drive/MyDrive/TFG Teleco/RANKINGS

Markdown página padre:
/content/drive/MyDrive/TFG Teleco/RANKINGS/rankings.md

Carpeta de recursos:
/content/drive/MyDrive/TFG Teleco/RANKINGS/rankings

OK: estructura de directorios preparada.



In [4]:
# ==========================================================
# BLOQUE 4. EXTRACCIÓN DE LOS RANKINGS
#
# Esta página NO tiene un patrón de href fijo como
# "/entidades/..." en Servicios, ni carga contenido por JS.
# Es una secuencia de tarjetas, cada una con:
#
#   - Un encabezado (h2/h3/h4...) que enlaza a la noticia
#     UPV que desarrolla ese ranking.
#   - Un párrafo (u otro texto suelto) describiéndolo.
#   - Un enlace a la web externa del ranking (dominio
#     distinto de upv.es), normalmente marcado con el icono
#     "abre en nueva ventana".
#
# Estrategia: recorremos los encabezados del contenedor
# principal; para cada uno que enlace a una noticia de UPV
# (patrón /noticias-upv/), tomamos como "bloque" todo lo que
# hay hasta el siguiente encabezado, y de ahí sacamos la
# descripción (primer párrafo con texto) y el enlace externo
# (primer <a> cuyo dominio no sea upv.es).
#
# AVISO: si tras ejecutar ves que faltan tarjetas o se cuela
# ruido (p.ej. el bloque de cabecera VA/EN), ajusta
# PATRON_NOTICIA / TAGS_ENCABEZADO según el HTML real.
# ==========================================================

PATRON_NOTICIA = re.compile(r"/noticias-upv/", re.IGNORECASE)

TAGS_ENCABEZADO = ["h1", "h2", "h3", "h4", "h5", "h6"]


# ----------------------------------------------------------
# 1. Descargar la página
# ----------------------------------------------------------

respuesta = requests.get(
    URL_RANKINGS,
    headers=HEADERS,
    timeout=30
)

respuesta.raise_for_status()

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)


# ----------------------------------------------------------
# 2. Localizar el contenedor principal
# ----------------------------------------------------------

contenedor_principal = (
    soup.find(id="smooth-wrapper")
    or soup.find("main")
    or soup.body
)

if contenedor_principal is None:

    raise Exception(
        "No se ha encontrado el contenedor principal de la página."
    )


# ----------------------------------------------------------
# 3. Localizar los encabezados que son "tarjeta de ranking"
# ----------------------------------------------------------

encabezados = contenedor_principal.find_all(TAGS_ENCABEZADO)

encabezados_ranking = []

for encabezado in encabezados:

    enlace = encabezado.find("a", href=True)

    if enlace is None:
        continue

    href = enlace.get("href", "")

    if not PATRON_NOTICIA.search(href):
        continue

    encabezados_ranking.append(
        (encabezado, enlace)
    )


# ----------------------------------------------------------
# 4. Para cada tarjeta, extraer título, descripción y
#    enlace externo, recorriendo los hermanos hasta el
#    siguiente encabezado
# ----------------------------------------------------------

def es_url_externa(url):

    try:

        dominio = urlparse(url).netloc.lower()

    except Exception:

        return False

    if not dominio:
        return False

    return "upv.es" not in dominio


recursos = []

for encabezado, enlace_titulo in encabezados_ranking:

    titulo = limpiar_texto(
        enlace_titulo.get_text(" ", strip=True)
    ) or limpiar_texto(
        encabezado.get_text(" ", strip=True)
    )

    url_noticia = normalizar_url(
        enlace_titulo.get("href", ""),
        URL_RANKINGS
    )

    descripcion = ""
    url_externa = ""

    for hermano in encabezado.find_next_siblings():

        # Nos detenemos al llegar al siguiente encabezado
        # (empieza la siguiente tarjeta).
        if hermano.name in TAGS_ENCABEZADO:
            break

        # Descripción: primer bloque de texto "suelto"
        # (párrafo o similar) con contenido razonable.
        if not descripcion:

            texto_bloque = limpiar_texto(
                hermano.get_text(" ", strip=True)
            )

            if hermano.name in {"p", "div", "span"} and len(texto_bloque) > 15:
                descripcion = texto_bloque

        # Enlace externo: el primer <a> de este bloque cuyo
        # dominio no sea upv.es.
        if not url_externa:

            for enlace_candidato in hermano.find_all("a", href=True):

                href_candidato = normalizar_url(
                    enlace_candidato.get("href", ""),
                    URL_RANKINGS
                )

                if es_url_externa(href_candidato):
                    url_externa = href_candidato
                    break

    if not titulo or not es_url_valida(url_noticia):
        continue

    recursos.append(
        crear_elemento(
            titulo=titulo,
            descripcion=descripcion,
            url=url_noticia,
            tipo="ranking",
            url_externa=url_externa
        )
    )

recursos = deduplicar_lista(
    recursos,
    clave="url"
)


# ----------------------------------------------------------
# 5. Construir el JSON
# ----------------------------------------------------------

seccion = crear_seccion(
    "Rankings",
    "rankings"
)

seccion["elementos"] = recursos

json_rankings = {

    "titulo": "La UPV en los rankings",

    "url": URL_RANKINGS,

    "tipo": "padre",

    "secciones": [seccion]
}


# ----------------------------------------------------------
# 6. Mostrar resultado
# ----------------------------------------------------------

print()
print("=" * 70)
print("ESTRUCTURA EXTRAÍDA - RANKINGS")
print("=" * 70)

print()

for elemento in seccion["elementos"]:

    print(f"  - {elemento['titulo']}")
    print(f"    {elemento['url']}")

    if elemento.get("url_externa"):
        print(f"    (externa) {elemento['url_externa']}")

print()
print("Total de rankings encontrados:", len(seccion["elementos"]))

print()
print("=" * 70)


ESTRUCTURA EXTRAÍDA - RANKINGS

  - LA UPV EN EL RANKING DE SHANGHÁI
    https://www.upv.es/noticias-upv/noticia-15353-arwu-2025-es.html
  - QS: Top 500 mundial por 15º año consecutivo
    http://preview.upv.es/noticias-upv/noticia-15901-qs-world-unive-es.html
    (externa) https://www.topuniversities.com/world-university-rankings?items_per_page=100
  - THE: La UPV, universidad con mayor impacto social y económico de España
    https://www.upv.es/noticias-upv/noticia-15919-the-sustainabi-es.html
    (externa) https://www.timeshighereducation.com/impactrankings
  - Greenmetric: Top 5 nacional y única politécnica española
    https://www.upv.es/noticias-upv/noticia-15603-ui-greenmetric-es.html
    (externa) https://uigreenmetric.com/rankings/university/overall-rankings-2025
  - Shanghái por materias: Top400 en 17 disciplinas
    https://www.upv.es/noticias-upv/noticia-15590-shanghai-por-m-es.html
    (externa) https://www.shanghairanking.com/rankings/gras/2025
  - QS por materias: Nº1 d

In [5]:
# ==========================================================
# BLOQUE 5. GUARDAR EL JSON
# ==========================================================

with open(
    RUTA_JSON,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        json_rankings,
        archivo,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 70)
print("JSON GUARDADO CORRECTAMENTE")
print("=" * 70)

print()
print("Archivo JSON:")
print(RUTA_JSON)

print()
print(
    "Número total de recursos:",
    sum(
        len(seccion["elementos"])
        for seccion in json_rankings["secciones"]
    )
)

print()
print("=" * 70)


JSON GUARDADO CORRECTAMENTE

Archivo JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/rankings.json

Número total de recursos: 7



In [6]:
# ==========================================================
# BLOQUE 6. GENERACIÓN DE MARKDOWN — RANKINGS
#
# Genera, en una carpeta nueva "RANKINGS" dentro de la ruta
# del programa:
#
#   - rankings.md            (página padre)
#   - un .md por cada ranking del JSON (visita la noticia
#     UPV real y extrae el contenido)
#
# Reutiliza el mismo motor de limpieza validado con
# "institución" y "servicios": recorte desde el <h1> real,
# filtrado de menú/pie/breadcrumbs/RRSS genéricos de upv.es,
# corte en widgets de plantilla y deduplicado global.
# ==========================================================

import os
import re
import time
import unicodedata
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


# ==========================================================
# 0. CONFIGURACIÓN DE METADATOS Y UMBRALES
# ==========================================================

FUENTE = "UPV"
CATEGORIA = "rankings"
NIVEL = "institucional"
PADRE_SLUG = "rankings"

UMBRAL_PALABRAS_POCO_CONTENIDO = 60
MAX_ENLACES_HIJOS = 5
MAX_CARACTERES_FRAGMENTO_HIJO = 800

TAGS_TITULO = {"h1", "h2", "h3", "h4", "h5", "h6"}

TAGS_BLOQUE = {
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p", "li", "div", "ul", "ol",
    "table", "section", "article", "blockquote"
}

TAGS_CANDIDATAS = [
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p", "li", "div", "span", "a"
]


# ==========================================================
# 1. FUNCIONES AUXILIARES (idénticas a Servicios)
# ==========================================================

def nombre_archivo_markdown(titulo):
    return normalizar_identificador(titulo) + ".md"


def extraer_texto_limpio(elemento):
    if elemento is None:
        return ""
    texto = elemento.get_text(" ", strip=True)
    return limpiar_texto(texto)


def descargar_soup(url):
    """
    Descarga una página. Devuelve (soup, es_html).
    Si el recurso no es HTML (PDF, vídeo, etc.) es_html
    será False y soup será None.
    """

    respuesta = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    respuesta.raise_for_status()

    content_type = respuesta.headers.get("Content-Type", "")

    if "html" not in content_type.lower():
        return None, False

    return (
        BeautifulSoup(respuesta.text, "html.parser"),
        True
    )


def limpiar_contenido_html(soup):
    for elemento in soup.find_all(
        ["script", "style", "noscript", "svg", "nav", "footer", "header"]
    ):
        elemento.decompose()

    return soup


# ==========================================================
# 2. EXTRACCIÓN DE CONTENIDO (hojas + enlaces) — sin cambios
# ==========================================================

def es_hoja_de_contenido(tag):

    if tag.name not in TAGS_CANDIDATAS:
        return False

    if tag.name == "a":
        return True

    descendientes_bloque = tag.find_all(TAGS_BLOQUE, recursive=True)

    return len(descendientes_bloque) == 0


def texto_markdown_de_elemento(tag, url_pagina):

    partes = []

    for nodo in tag.children:

        nombre_nodo = getattr(nodo, "name", None)

        if nombre_nodo == "a":

            texto_enlace = extraer_texto_limpio(nodo)
            href = (nodo.get("href") or "").strip()

            if not texto_enlace:
                continue

            if href and not href.startswith(("javascript:", "#")):

                href_absoluta = urljoin(url_pagina, href)
                href_absoluta = resolver_url_menu_antiguo(href_absoluta)

                partes.append(f"[{texto_enlace}]({href_absoluta})")

            else:

                partes.append(texto_enlace)

        elif nombre_nodo is not None:

            texto = extraer_texto_limpio(nodo)

            if texto:
                partes.append(texto)

        else:

            texto = limpiar_texto(str(nodo))

            if texto:
                partes.append(texto)

    return limpiar_texto(" ".join(partes))


def extraer_bloques_contenido(contenedor, url_pagina):

    lineas = []
    linea_anterior = None

    for tag in contenedor.find_all(TAGS_CANDIDATAS, recursive=True):

        if not es_hoja_de_contenido(tag):
            continue

        if tag.name == "a":

            texto = extraer_texto_limpio(tag)
            href = (tag.get("href") or "").strip()

            if not texto:
                continue

            if href and not href.startswith(("javascript:", "#")):

                href_absoluta = urljoin(url_pagina, href)
                href_absoluta = resolver_url_menu_antiguo(href_absoluta)
                linea = f"[{texto}]({href_absoluta})"

            else:

                linea = texto

        else:

            texto = texto_markdown_de_elemento(tag, url_pagina)

            if not texto:
                continue

            if tag.name in TAGS_TITULO:

                nivel = int(tag.name[1])
                linea = ("#" * nivel) + " " + texto

            elif tag.name == "li":

                linea = f"- {texto}"

            else:

                linea = texto

        if linea == linea_anterior:
            continue

        lineas.append(linea)
        linea_anterior = linea

    return lineas


def contar_palabras(lineas):
    return sum(len(linea.split()) for linea in lineas)


# ==========================================================
# 2bis. FILTRADO DE PLANTILLA (idéntico: mismo menú/pie
# en todo upv.es, sirve para cualquier sección)
# ==========================================================

def normalizar_para_comparar(texto):

    texto = texto.strip().lower()
    texto = texto.strip("¡¿!?: ")

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


TEXTOS_BOILERPLATE = {
    "accesibilidad", "mapa web", "buscar", "directorio",
    "iniciar sesion", "emergencias", "inicio upv",
    "admision", "estudios", "investigacion", "organizacion",
    "comunidad upv",
    "admision a grado", "admision a master", "admision a doctorado",
    "internacional",
    "estudios de grado", "estudios de posgrado", "oferta academica",
    "estructuras de investigacion", "iniciativas de i+d+i", "innovacion",
    "la institucion", "vida universitaria", "escuelas y facultades",
    "departamentos", "servicios universitarios",
    "estudiante", "pas, pdi y pi", "ptgas, pdi y pi", "prensa", "titulados",
    "alumni upv", "orientador",
    "como llegar", "planos", "planos 2d", "contacto",
    "habla con nosotros",
    "tienes dudas", "contacta con nosotros",
    "quieres enviar una sugerencia, queja o felicitacion",
    "no has encontrado lo que buscas",
    "dinos que opinas", "consultanos",
    "sala de prensa", "noticias de la upv", "buscar un cargo docente",
    "area de comunicacion", "transparencia", "perfil del contratante",
    "aviso legal", "politica de cookies", "politica de privacidad",
    "gestion de cookies", "descarga nuestras apps",
    "idioma", "idioma · language", "language",
    "valencia", "valencian", "english", "castellano",
    "cercar", "search", "directory", "directori",
    "contacte", "contact",
    "otros", "donde estamos", "¿donde estamos?",
    "webs relacionadas",
    # Específicos de la página de rankings:
    "va", "en", "web (abre en nueva ventana)",
}

TITULOS_CORTE_PLANTILLA = {
    "esto te interesa", "recursos", "instalaciones", "media",
}


def es_linea_boilerplate(linea):

    texto_plano = re.sub(r"^#+\s*", "", linea)
    texto_plano = re.sub(r"^-\s*", "", texto_plano)

    match_enlace = re.match(r"^\[([^\]]+)\]\([^)]+\)$", texto_plano.strip())
    if match_enlace:
        texto_plano = match_enlace.group(1)

    normalizado = normalizar_para_comparar(texto_plano)

    if normalizado in TEXTOS_BOILERPLATE:
        return True

    if "universitat politecnica de valencia" in normalizado and "©" in linea:
        return True

    if re.match(r"^tel\.?\s*\(?\+?34", normalizado):
        return True

    if "::" in linea:
        return True

    if linea.startswith("#") and normalizado in TITULOS_CORTE_PLANTILLA:
        return True

    return False


def recortar_desde_primer_h1(lineas):

    for indice, linea in enumerate(lineas):

        if linea.startswith("# "):
            return lineas[indice:]

    return lineas


def recortar_en_titulo_plantilla(lineas):

    for indice, linea in enumerate(lineas):

        if not linea.startswith("#"):
            continue

        texto_titulo = re.sub(r"^#+\s*", "", linea)
        normalizado = normalizar_para_comparar(texto_titulo)

        if normalizado in TITULOS_CORTE_PLANTILLA:
            return lineas[:indice]

    return lineas


def deduplicar_global(lineas):

    resultado = []
    vistos = set()

    for linea in lineas:

        if linea in vistos:
            continue

        vistos.add(linea)
        resultado.append(linea)

    return resultado


def limpiar_lineas_finales(lineas, recortar_h1=True):

    if recortar_h1:
        lineas = recortar_desde_primer_h1(lineas)

    lineas = recortar_en_titulo_plantilla(lineas)

    lineas = [
        linea for linea in lineas
        if not es_linea_boilerplate(linea)
    ]

    lineas = deduplicar_global(lineas)

    return lineas


# ==========================================================
# 3. RUTAS DE SALIDA (carpeta plana "RANKINGS")
# ==========================================================

CARPETA_RANKINGS = os.path.join(ruta_programa, "RANKINGS")
os.makedirs(CARPETA_RANKINGS, exist_ok=True)

RUTA_MARKDOWN_PADRE_RANKINGS = os.path.join(
    CARPETA_RANKINGS, "rankings.md"
)

if "RUTA_JSON" not in dir():
    RUTA_JSON = os.path.join(CARPETA_JSON, "rankings.json")

URL_RANKINGS = "https://www.upv.es/rankings/index.html"


# ==========================================================
# 4. CARGAR EL JSON (aquí no hace falta filtrar "fichas
# falsas" como en Servicios: cada elemento ya viene de una
# tarjeta de ranking real con enlace a noticia UPV; solo se
# hace un deduplicado defensivo por URL)
# ==========================================================

with open(RUTA_JSON, "r", encoding="utf-8") as archivo:
    json_rankings = json.load(archivo)

seccion_rankings = json_rankings["secciones"][0]
elementos_originales = seccion_rankings["elementos"]

elementos_limpios = []
urls_vistas = set()

for elemento in elementos_originales:

    url = elemento.get("url", "")

    if not url or url in urls_vistas:
        continue

    urls_vistas.add(url)
    elementos_limpios.append(elemento)

seccion_rankings["elementos"] = elementos_limpios

print()
print("=" * 70)
print("CARGA DEL JSON DE RANKINGS")
print("=" * 70)
print()
print("Elementos originales en el JSON:", len(elementos_originales))
print("Rankings finales a procesar:", len(elementos_limpios))
print("=" * 70)


# ==========================================================
# 5. EXPANSIÓN DE ENLACES HIJOS (páginas con poco contenido)
#    — idéntico a Servicios
# ==========================================================

def es_url_valida_para_expandir(href, url_pagina, urls_ya_usadas):

    if not href:
        return False

    href = href.strip()

    if href.startswith(("mailto:", "tel:", "javascript:", "#")):
        return False

    absoluta = urljoin(url_pagina, href).split("#")[0]

    if absoluta == url_pagina.split("#")[0]:
        return False

    if absoluta in urls_ya_usadas:
        return False

    dominio = urlparse(absoluta).netloc

    if "upv.es" not in dominio:
        return False

    return True


PATRONES_URL_EXCLUIDOS_HIJOS = [
    r"/bin2/tipoacc/",
    r"sic_mag\.MetaBus",
    r"/plano/plano-2d",
    r"/otros/como-llegar",
    r"index-va\.html?$",
    r"index-en\.html?$",
    r"index-i\.html?$",
    r"index-v\.html?$",
    r"/otros/accesibilidad",
    r"/otros/mapa-web",
    r"/otros/contacto",
]


def es_directorio_generico_de_personas(href):

    if "sic_per.Busca_Persona" not in href:
        return False

    return "P_SG=" not in href and "P_CARGOS=" not in href


def es_variante_de_la_misma_pagina(href_absoluta, url_pagina):

    def codigo_pagina(url):
        return urlparse(url).path.rstrip("/").lower()

    if codigo_pagina(href_absoluta) != codigo_pagina(url_pagina):
        return False

    return bool(re.search(r"/index\w*\.html?$", href_absoluta, flags=re.IGNORECASE))


def resolver_url_menu_antiguo(url_absoluta):

    coincidencia = re.search(r"menu_url\w*\.html\?(//.+)$", url_absoluta, flags=re.IGNORECASE)

    if not coincidencia:
        return url_absoluta

    interno = coincidencia.group(1)

    if interno.startswith("//"):
        interno = "https:" + interno

    return interno


def extraer_texto_y_url_de_linea_markdown(linea):

    coincidencia = re.match(r"^\[([^\]]*)\]\(([^)]+)\)$", linea.strip())

    if not coincidencia:
        return None, None

    return coincidencia.group(1), coincidencia.group(2)


def es_autorreferencia_o_accesibilidad(linea, url_pagina):

    texto, href = extraer_texto_y_url_de_linea_markdown(linea)

    if href is None:
        return False

    if len(texto.strip()) <= 2:
        return True

    return es_variante_de_la_misma_pagina(href, url_pagina)


TEXTOS_EXCLUIDOS_HIJOS = {
    "valencia", "valencia language", "valencian", "english",
    "castellano", "cercar", "search", "directory", "directori",
    "contacte", "contact", "idioma", "language", "idioma language",
}

TEXTOS_PRIORITARIOS_HIJOS = [
    "informacion general", "quienes somos", "presentacion",
    "equipo directivo", "webs relacionadas", "servicios",
    "tramites", "memoria", "funciones", "organigrama",
]


def es_enlace_hijo_util(texto, href, url_pagina):

    texto_normalizado = normalizar_para_comparar(texto)

    if len(texto.strip()) <= 2:
        return False

    if texto_normalizado in TEXTOS_BOILERPLATE:
        return False

    if texto_normalizado in TEXTOS_EXCLUIDOS_HIJOS:
        return False

    if es_directorio_generico_de_personas(href):
        return False

    absoluta = urljoin(url_pagina, href).split("#")[0]
    absoluta = resolver_url_menu_antiguo(absoluta)

    if es_variante_de_la_misma_pagina(absoluta, url_pagina):
        return False

    for patron in PATRONES_URL_EXCLUIDOS_HIJOS:
        if re.search(patron, href, flags=re.IGNORECASE):
            return False

    return True


def obtener_enlaces_hijos(contenedor, url_pagina, maximo=MAX_ENLACES_HIJOS):

    candidatos = []
    urls_vistas_local = set()

    for a in contenedor.find_all("a", href=True):

        href = a["href"]

        if not es_url_valida_para_expandir(href, url_pagina, urls_vistas_local):
            continue

        texto = extraer_texto_limpio(a)

        if not texto:
            continue

        if not es_enlace_hijo_util(texto, href, url_pagina):
            continue

        absoluta = urljoin(url_pagina, href).split("#")[0]
        absoluta = resolver_url_menu_antiguo(absoluta)

        urls_vistas_local.add(absoluta)
        candidatos.append((texto, absoluta))

    def prioridad(candidato):
        texto_normalizado = normalizar_para_comparar(candidato[0])
        return 0 if texto_normalizado in TEXTOS_PRIORITARIOS_HIJOS else 1

    candidatos.sort(key=prioridad)

    return candidatos[:maximo]


def resumir_pagina_hija(url):

    try:

        soup, es_html = descargar_soup(url)

        if not es_html:
            return None

        soup = limpiar_contenido_html(soup)

        contenedor = (
            soup.find(id="smooth-wrapper")
            or soup.find("main")
            or soup.body
        )

        if contenedor is None:
            return None

        lineas = extraer_bloques_contenido(contenedor, url)
        lineas = limpiar_lineas_finales(lineas)
        lineas = [l for l in lineas if not es_autorreferencia_o_accesibilidad(l, url)]

        fragmento = "\n\n".join(lineas)

        if len(fragmento) > MAX_CARACTERES_FRAGMENTO_HIJO:
            fragmento = fragmento[:MAX_CARACTERES_FRAGMENTO_HIJO].rstrip() + "…"

        return fragmento or None

    except Exception:

        return None


# ==========================================================
# 6. METADATOS YAML — ahora incluye url_externa si existe
# ==========================================================

def generar_yaml_metadatos(seccion_id, recurso, tipo_documento="recurso", tipo_recurso="ranking"):

    campos = [
        ("fuente", FUENTE),
        ("categoria", CATEGORIA),
        ("nivel", NIVEL),
        ("tipo_documento", tipo_documento),
        ("tipo_recurso", tipo_recurso),
        ("padre", PADRE_SLUG),
        ("seccion", seccion_id),
        ("url", recurso.get("url", "")),
    ]

    if recurso.get("url_externa"):
        campos.append(("url_externa", recurso["url_externa"]))

    lineas = [f"{clave}: {valor}" for clave, valor in campos]

    return "---\n" + "\n\n".join(lineas) + "\n---\n"


# ==========================================================
# 7. GENERAR MARKDOWN DE CADA RANKING
#    (misma lógica que generar_markdown_recurso en Servicios)
# ==========================================================

def generar_markdown_recurso(recurso, carpeta, seccion_id):

    titulo = recurso.get("titulo", "")
    url = recurso.get("url", "")

    if not titulo or not url:
        return False

    print()
    print(f"  Extrayendo: {titulo}")
    print(f"  URL: {url}")

    try:

        soup, es_html = descargar_soup(url)

        yaml_metadatos = generar_yaml_metadatos(seccion_id, recurso)

        if not es_html:

            markdown = (
                f"{yaml_metadatos}\n"
                f"# {titulo}\n\n"
                f"**URL:** {url}\n\n"
                f"_Este recurso no es una página HTML estándar "
                f"(por ejemplo, un PDF o un vídeo). "
                f"Consulta el contenido directamente en la URL indicada._\n"
            )

            nombre_archivo = nombre_archivo_markdown(titulo)
            ruta_archivo = os.path.join(carpeta, nombre_archivo)

            with open(ruta_archivo, "w", encoding="utf-8") as archivo:
                archivo.write(markdown)

            print(f"  OK (no HTML): {ruta_archivo}")

            return True

        soup = limpiar_contenido_html(soup)

        contenido = (
            soup.find(id="smooth-wrapper")
            or soup.find("main")
            or soup.body
        )

        if contenido is None:
            print("  AVISO: no se ha encontrado contenido.")
            return False

        lineas_contenido = extraer_bloques_contenido(contenido, url)
        lineas_contenido = limpiar_lineas_finales(lineas_contenido)
        lineas_contenido = [
            l for l in lineas_contenido
            if not es_autorreferencia_o_accesibilidad(l, url)
        ]

        if not lineas_contenido:
            print("  AVISO: contenido vacío.")
            return False

        if contar_palabras(lineas_contenido) < UMBRAL_PALABRAS_POCO_CONTENIDO:

            enlaces_hijos = obtener_enlaces_hijos(contenido, url)

            if enlaces_hijos:

                lineas_contenido.append("## Información relacionada")

                for texto_enlace, url_hija in enlaces_hijos:

                    time.sleep(0.5)

                    fragmento = resumir_pagina_hija(url_hija)

                    lineas_contenido.append(f"### {texto_enlace}")
                    lineas_contenido.append(f"**URL:** {url_hija}")

                    if fragmento:
                        lineas_contenido.append(fragmento)

        markdown_contenido = "\n\n".join(lineas_contenido)

        descripcion = recurso.get("descripcion", "").strip()

        bloque_descripcion = (
            f"**Descripción breve:** {descripcion}\n\n"
            if descripcion else ""
        )

        bloque_url_externa = (
            f"**Fuente del ranking:** {recurso['url_externa']}\n\n"
            if recurso.get("url_externa") else ""
        )

        markdown = (
            f"{yaml_metadatos}\n"
            f"# {titulo}\n\n"
            f"**URL:** {url}\n\n"
            f"{bloque_descripcion}"
            f"{bloque_url_externa}"
            f"{markdown_contenido}\n"
        )

        nombre_archivo = nombre_archivo_markdown(titulo)
        ruta_archivo = os.path.join(carpeta, nombre_archivo)

        with open(ruta_archivo, "w", encoding="utf-8") as archivo:
            archivo.write(markdown)

        print(f"  OK: {ruta_archivo}")

        return True

    except Exception as error:

        print(f"  ERROR: {error}")
        return False


# ==========================================================
# 8. GENERAR MARKDOWN DE LA PÁGINA PADRE
#
# Filtramos del padre los enlaces a /noticias-upv/ (ya son
# documentos propios, uno por cada ranking) para no duplicar
# el listado completo dentro del padre.
# ==========================================================

def es_enlace_a_noticia(linea):
    return bool(re.search(r"\]\(https?://(www\.)?upv\.es/noticias-upv/", linea))


def es_ruido_listado_rankings(linea):

    if es_enlace_a_noticia(linea):
        return True

    return False


def generar_markdown_padre_rankings():

    print()
    print("=" * 70)
    print("GENERANDO MARKDOWN DE LA PÁGINA PADRE (RANKINGS)")
    print("=" * 70)

    soup, es_html = descargar_soup(URL_RANKINGS)
    soup = limpiar_contenido_html(soup)

    contenedor = (
        soup.find(id="smooth-wrapper")
        or soup.find("main")
        or soup.body
    )

    if contenedor is None:
        raise Exception("No se ha encontrado el contenedor principal.")

    lineas = extraer_bloques_contenido(contenedor, URL_RANKINGS)
    lineas = limpiar_lineas_finales(lineas)

    lineas = [linea for linea in lineas if not es_ruido_listado_rankings(linea)]
    lineas = deduplicar_global(lineas)

    yaml_metadatos = generar_yaml_metadatos(
        seccion_id="rankings",
        recurso={"url": URL_RANKINGS},
        tipo_documento="padre",
        tipo_recurso="informacion",
    )

    markdown = (
        f"{yaml_metadatos}\n"
        + "\n\n".join(lineas)
        + "\n"
    )

    with open(RUTA_MARKDOWN_PADRE_RANKINGS, "w", encoding="utf-8") as archivo:
        archivo.write(markdown)

    print()
    print("OK: Markdown padre generado:")
    print(RUTA_MARKDOWN_PADRE_RANKINGS)

    return markdown


# ==========================================================
# 9. EJECUCIÓN
# ==========================================================

markdown_padre_rankings = generar_markdown_padre_rankings()

total_rankings = 0
rankings_correctos = 0
rankings_error = 0

print()
print("=" * 70)
print("GENERANDO MARKDOWNS DE RANKINGS")
print("=" * 70)

for recurso in seccion_rankings["elementos"]:

    total_rankings += 1

    if generar_markdown_recurso(recurso, CARPETA_RANKINGS, seccion_rankings["id"]):
        rankings_correctos += 1
    else:
        rankings_error += 1


# ==========================================================
# 10. RESUMEN FINAL
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN DE MARKDOWN FINALIZADA")
print("=" * 70)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE_RANKINGS)

print()
print("Rankings procesados:", total_rankings)
print("Markdowns generados:", rankings_correctos)
print("Errores:", rankings_error)

print()
print("Total de documentos en la carpeta RANKINGS:", rankings_correctos + 1)

print()
print("Carpeta de salida:")
print(CARPETA_RANKINGS)

print()
print("=" * 70)


CARGA DEL JSON DE RANKINGS

Elementos originales en el JSON: 7
Rankings finales a procesar: 7

GENERANDO MARKDOWN DE LA PÁGINA PADRE (RANKINGS)

OK: Markdown padre generado:
/content/drive/MyDrive/TFG Teleco/RANKINGS/rankings.md

GENERANDO MARKDOWNS DE RANKINGS

  Extrayendo: LA UPV EN EL RANKING DE SHANGHÁI
  URL: https://www.upv.es/noticias-upv/noticia-15353-arwu-2025-es.html
  OK: /content/drive/MyDrive/TFG Teleco/RANKINGS/la_upv_en_el_ranking_de_shanghai.md

  Extrayendo: QS: Top 500 mundial por 15º año consecutivo
  URL: http://preview.upv.es/noticias-upv/noticia-15901-qs-world-unive-es.html
  ERROR: 401 Client Error: Unauthorized for url: http://preview.upv.es/noticias-upv/noticia-15901-qs-world-unive-es.html

  Extrayendo: THE: La UPV, universidad con mayor impacto social y económico de España
  URL: https://www.upv.es/noticias-upv/noticia-15919-the-sustainabi-es.html
  OK: /content/drive/MyDrive/TFG Teleco/RANKINGS/the_la_upv_universidad_con_mayor_impacto_social_y_economico_de